# Spring of Code - Artificial Intelligence
## Week 02: Descriptive Statistics and Probability
### Day 03: Correlation & Probability

In this notebook, we will learn about **Correlation & Probability** using Python.

In [1]:
import math
import random
from scipy.stats import pearsonr

---
Why Statistics & Probability in AI/ML?

Before building any ML model, we must **understand the data**.
Statistics answers: *What does my data look like?*  
Probability answers: *How uncertain am I about an outcome?*

**Let's start with the most basic descriptive statistics — computed by hand in Python.**

In [2]:
# A sample dataset: exam scores of 15 students
scores = [55, 72, 68, 91, 85, 60, 77, 83, 90, 65, 70, 88, 75, 62, 95]

n = len(scores)

# Mean 
mean = sum(scores) / n

# Median
sorted_scores = sorted(scores)
mid = n // 2
if n % 2 == 0:
    median = (sorted_scores[mid - 1] + sorted_scores[mid]) / 2
else:
    median = sorted_scores[mid]

# Mode 
frequency = {}
for value in scores:
    if value in frequency:
        frequency[value] += 1
    else:
        frequency[value] = 1

max_count = 0
mode = None

for key in frequency:
    if frequency[key] > max_count:
        max_count = frequency[key]
        mode = key

# Variance & Standard Deviation
variance = sum((x - mean) ** 2 for x in scores) / n      # population variance
std_dev  = math.sqrt(variance)

# Range
data_range = max(scores) - min(scores)

# IQR (Interquartile Range)
q1 = sorted_scores[int(25/100 * n)]        # 25th percentile (approximate)
q3 = sorted_scores[int(75/100 * n)]  # 75th percentile (approximate)
iqr = q3 - q1

print('=== Descriptive Statistics: Exam Scores ===')
print(f'Data (sorted): {sorted_scores}')
print()
print(f'Mean     (average)           = {mean:.2f}')
print(f'Median   (middle value)      = {median}')
print(f'Mode     (most frequent)     = {mode}')
print(f'Std Dev  (spread)            = {std_dev:.2f}')
print(f'Variance                     = {variance:.2f}')
print(f'Range    (max - min)         = {data_range}')
print(f'Min                          = {min(scores)}')
print(f'Max                          = {max(scores)}')
print(f'Q1 (25th percentile)         = {q1}')
print(f'Q3 (75th percentile)         = {q3}')
print(f'IQR (Q3 - Q1)                = {iqr}')

=== Descriptive Statistics: Exam Scores ===
Data (sorted): [55, 60, 62, 65, 68, 70, 72, 75, 77, 83, 85, 88, 90, 91, 95]

Mean     (average)           = 75.73
Median   (middle value)      = 75
Mode     (most frequent)     = 55
Std Dev  (spread)            = 12.07
Variance                     = 145.80
Range    (max - min)         = 40
Min                          = 55
Max                          = 95
Q1 (25th percentile)         = 65
Q3 (75th percentile)         = 88
IQR (Q3 - Q1)                = 23


### UNDERSTANDING MEAN vs MEDIAN
Outliers affect mean much more than median

In [3]:
# Normal salaries in a company
salaries = [30000, 32000, 35000, 28000, 40000, 33000, 31000]

mean_normal   = sum(salaries) / len(salaries)
sorted_sal    = sorted(salaries)
median_normal = sorted_sal[len(sorted_sal) // 2]

# Now add the CEO's salary (outlier)
salaries_with_ceo = salaries + [2000000]

mean_with_ceo   = sum(salaries_with_ceo) / len(salaries_with_ceo)
sorted_ceo      = sorted(salaries_with_ceo)
median_with_ceo = sorted_ceo[len(sorted_ceo) // 2]

print('=== Effect of Outliers on Mean vs Median ===')
print()
print('Without CEO salary:')
print(f'  Mean   = ${mean_normal:,.0f}')
print(f'  Median = ${median_normal:,.0f}')
print()
print('After adding CEO salary ($2,000,000):')
print(f'  Mean   = ${mean_with_ceo:,.0f}  ← JUMPED drastically!')
print(f'  Median = ${median_with_ceo:,.0f}  ← barely changed')
print()
print('Lesson: Use MEDIAN when data has outliers (e.g. income, house prices)')
print('           Use MEAN when data is symmetric with no extreme outliers')

=== Effect of Outliers on Mean vs Median ===

Without CEO salary:
  Mean   = $32,714
  Median = $32,000

After adding CEO salary ($2,000,000):
  Mean   = $278,625  ← JUMPED drastically!
  Median = $33,000  ← barely changed

Lesson: Use MEDIAN when data has outliers (e.g. income, house prices)
           Use MEAN when data is symmetric with no extreme outliers


---
##  Correlation Coefficient

Correlation tells us **how strongly two variables move together**.

| Value | Meaning |
|-------|---------|
| +1.0  | Perfect positive relationship |
| +0.7 to +1.0 | Strong positive |
| +0.3 to +0.7 | Moderate positive |
| -0.3 to +0.3 | Weak / No relationship |
| -0.7 to -0.3 | Moderate negative |
| -1.0  | Perfect negative relationship |

In [4]:
with open('advertising.csv', 'r') as file:
    lines = file.readlines()
lines

['TV,Radio,Newspaper,Sales\n',
 '230.1,37.8,69.2,22.1\n',
 '44.5,39.3,45.1,10.4\n',
 '17.2,45.9,69.3,12\n',
 '151.5,41.3,58.5,16.5\n',
 '180.8,10.8,58.4,17.9\n',
 '8.7,48.9,75,7.2\n',
 '57.5,32.8,23.5,11.8\n',
 '120.2,19.6,11.6,13.2\n',
 '8.6,2.1,1,4.8\n',
 '199.8,2.6,21.2,15.6\n',
 '66.1,5.8,24.2,12.6\n',
 '214.7,24,4,17.4\n',
 '23.8,35.1,65.9,9.2\n',
 '97.5,7.6,7.2,13.7\n',
 '204.1,32.9,46,19\n',
 '195.4,47.7,52.9,22.4\n',
 '67.8,36.6,114,12.5\n',
 '281.4,39.6,55.8,24.4\n',
 '69.2,20.5,18.3,11.3\n',
 '147.3,23.9,19.1,14.6\n',
 '218.4,27.7,53.4,18\n',
 '237.4,5.1,23.5,17.5\n',
 '13.2,15.9,49.6,5.6\n',
 '228.3,16.9,26.2,20.5\n',
 '62.3,12.6,18.3,9.7\n',
 '262.9,3.5,19.5,17\n',
 '142.9,29.3,12.6,15\n',
 '240.1,16.7,22.9,20.9\n',
 '248.8,27.1,22.9,18.9\n',
 '70.6,16,40.8,10.5\n',
 '292.9,28.3,43.2,21.4\n',
 '112.9,17.4,38.6,11.9\n',
 '97.2,1.5,30,13.2\n',
 '265.6,20,0.3,17.4\n',
 '95.7,1.4,7.4,11.9\n',
 '290.7,4.1,8.5,17.8\n',
 '266.9,43.8,5,25.4\n',
 '74.7,49.4,45.7,14.7\n',
 '43.1,26.7

In [5]:
len(lines)

201

In [18]:
for line in lines[1:]:
    print(line.strip().split(',')[-1])

22.1
10.4
12
16.5
17.9
7.2
11.8
13.2
4.8
15.6
12.6
17.4
9.2
13.7
19
22.4
12.5
24.4
11.3
14.6
18
17.5
5.6
20.5
9.7
17
15
20.9
18.9
10.5
21.4
11.9
13.2
17.4
11.9
17.8
25.4
14.7
10.1
21.5
16.6
17.1
20.7
17.9
8.5
16.1
10.6
23.2
19.8
9.7
16.4
10.7
22.6
21.2
20.2
23.7
5.5
13.2
23.8
18.4
8.1
24.2
20.7
14
16
11.3
11
13.4
18.9
22.3
18.3
12.4
8.8
11
17
8.7
6.9
14.2
5.3
11
11.8
17.3
11.3
13.6
21.7
20.2
12
16
12.9
16.7
14
7.3
19.4
22.2
11.5
16.9
16.7
20.5
25.4
17.2
16.7
23.8
19.8
19.7
20.7
15
7.2
12
5.3
19.8
18.4
21.8
17.1
20.9
14.6
12.6
12.2
9.4
15.9
6.6
15.5
7
16.6
15.2
19.7
10.6
6.6
11.9
24.7
9.7
1.6
17.7
5.7
19.6
10.8
11.6
9.5
20.8
9.6
20.7
10.9
19.2
20.1
10.4
12.3
10.3
18.2
25.4
10.9
10.1
16.1
11.6
16.6
16
20.6
3.2
15.3
10.1
7.3
12.9
16.4
13.3
19.9
18
11.9
16.9
8
17.2
17.1
20
8.4
17.5
7.6
16.7
16.5
27
20.2
16.7
16.8
17.6
15.5
17.2
8.7
26.2
17.6
22.6
10.3
17.3
20.9
6.7
10.8
11.9
5.9
19.6
17.3
7.6
14
14.8
25.5
18.4


In [20]:
x = []
y = []
for line in lines[1:]:
    x.append(float(line.strip().split(',')[0]))
    y.append(float(line.strip().split(',')[-1]))

print(x)
print(y)


[230.1, 44.5, 17.2, 151.5, 180.8, 8.7, 57.5, 120.2, 8.6, 199.8, 66.1, 214.7, 23.8, 97.5, 204.1, 195.4, 67.8, 281.4, 69.2, 147.3, 218.4, 237.4, 13.2, 228.3, 62.3, 262.9, 142.9, 240.1, 248.8, 70.6, 292.9, 112.9, 97.2, 265.6, 95.7, 290.7, 266.9, 74.7, 43.1, 228.0, 202.5, 177.0, 293.6, 206.9, 25.1, 175.1, 89.7, 239.9, 227.2, 66.9, 199.8, 100.4, 216.4, 182.6, 262.7, 198.9, 7.3, 136.2, 210.8, 210.7, 53.5, 261.3, 239.3, 102.7, 131.1, 69.0, 31.5, 139.3, 237.4, 216.8, 199.1, 109.8, 26.8, 129.4, 213.4, 16.9, 27.5, 120.5, 5.4, 116.0, 76.4, 239.8, 75.3, 68.4, 213.5, 193.2, 76.3, 110.7, 88.3, 109.8, 134.3, 28.6, 217.7, 250.9, 107.4, 163.3, 197.6, 184.9, 289.7, 135.2, 222.4, 296.4, 280.2, 187.9, 238.2, 137.9, 25.0, 90.4, 13.1, 255.4, 225.8, 241.7, 175.7, 209.6, 78.2, 75.1, 139.2, 76.4, 125.7, 19.4, 141.3, 18.8, 224.0, 123.1, 229.5, 87.2, 7.8, 80.2, 220.3, 59.6, 0.7, 265.2, 8.4, 219.8, 36.9, 48.3, 25.6, 273.7, 43.0, 184.9, 73.4, 193.7, 220.5, 104.6, 96.2, 140.3, 240.1, 243.2, 38.0, 44.7, 280.7, 121.0

In [22]:
for i in range(1, len(lines)):
    print(lines[i].strip().split(',')[0])

230.1
44.5
17.2
151.5
180.8
8.7
57.5
120.2
8.6
199.8
66.1
214.7
23.8
97.5
204.1
195.4
67.8
281.4
69.2
147.3
218.4
237.4
13.2
228.3
62.3
262.9
142.9
240.1
248.8
70.6
292.9
112.9
97.2
265.6
95.7
290.7
266.9
74.7
43.1
228
202.5
177
293.6
206.9
25.1
175.1
89.7
239.9
227.2
66.9
199.8
100.4
216.4
182.6
262.7
198.9
7.3
136.2
210.8
210.7
53.5
261.3
239.3
102.7
131.1
69
31.5
139.3
237.4
216.8
199.1
109.8
26.8
129.4
213.4
16.9
27.5
120.5
5.4
116
76.4
239.8
75.3
68.4
213.5
193.2
76.3
110.7
88.3
109.8
134.3
28.6
217.7
250.9
107.4
163.3
197.6
184.9
289.7
135.2
222.4
296.4
280.2
187.9
238.2
137.9
25
90.4
13.1
255.4
225.8
241.7
175.7
209.6
78.2
75.1
139.2
76.4
125.7
19.4
141.3
18.8
224
123.1
229.5
87.2
7.8
80.2
220.3
59.6
0.7
265.2
8.4
219.8
36.9
48.3
25.6
273.7
43
184.9
73.4
193.7
220.5
104.6
96.2
140.3
240.1
243.2
38
44.7
280.7
121
197.6
171.3
187.8
4.1
93.9
149.8
11.7
131.7
172.5
85.7
188.4
163.5
117.2
234.5
17.9
206.8
215.4
284.3
50
164.5
19.6
168.4
222.4
276.9
248.4
170.2
276.7
165.6
156.6
218.5

In [34]:
x = []
y = []
for i in range(1, len(lines)):
    x.append(float(lines[i].strip().split(',')[0]))
    y.append(float(lines[i].strip().split(',')[-1]))

print(x)
print(y)

[230.1, 44.5, 17.2, 151.5, 180.8, 8.7, 57.5, 120.2, 8.6, 199.8, 66.1, 214.7, 23.8, 97.5, 204.1, 195.4, 67.8, 281.4, 69.2, 147.3, 218.4, 237.4, 13.2, 228.3, 62.3, 262.9, 142.9, 240.1, 248.8, 70.6, 292.9, 112.9, 97.2, 265.6, 95.7, 290.7, 266.9, 74.7, 43.1, 228.0, 202.5, 177.0, 293.6, 206.9, 25.1, 175.1, 89.7, 239.9, 227.2, 66.9, 199.8, 100.4, 216.4, 182.6, 262.7, 198.9, 7.3, 136.2, 210.8, 210.7, 53.5, 261.3, 239.3, 102.7, 131.1, 69.0, 31.5, 139.3, 237.4, 216.8, 199.1, 109.8, 26.8, 129.4, 213.4, 16.9, 27.5, 120.5, 5.4, 116.0, 76.4, 239.8, 75.3, 68.4, 213.5, 193.2, 76.3, 110.7, 88.3, 109.8, 134.3, 28.6, 217.7, 250.9, 107.4, 163.3, 197.6, 184.9, 289.7, 135.2, 222.4, 296.4, 280.2, 187.9, 238.2, 137.9, 25.0, 90.4, 13.1, 255.4, 225.8, 241.7, 175.7, 209.6, 78.2, 75.1, 139.2, 76.4, 125.7, 19.4, 141.3, 18.8, 224.0, 123.1, 229.5, 87.2, 7.8, 80.2, 220.3, 59.6, 0.7, 265.2, 8.4, 219.8, 36.9, 48.3, 25.6, 273.7, 43.0, 184.9, 73.4, 193.7, 220.5, 104.6, 96.2, 140.3, 240.1, 243.2, 38.0, 44.7, 280.7, 121.0

In [26]:
x = []
y = []
for line in lines[1:]:
    x.append(float(line.strip().split(',')[0]))
    y.append(float(line.strip().split(',')[1]))

print(x)
print(y)


[230.1, 44.5, 17.2, 151.5, 180.8, 8.7, 57.5, 120.2, 8.6, 199.8, 66.1, 214.7, 23.8, 97.5, 204.1, 195.4, 67.8, 281.4, 69.2, 147.3, 218.4, 237.4, 13.2, 228.3, 62.3, 262.9, 142.9, 240.1, 248.8, 70.6, 292.9, 112.9, 97.2, 265.6, 95.7, 290.7, 266.9, 74.7, 43.1, 228.0, 202.5, 177.0, 293.6, 206.9, 25.1, 175.1, 89.7, 239.9, 227.2, 66.9, 199.8, 100.4, 216.4, 182.6, 262.7, 198.9, 7.3, 136.2, 210.8, 210.7, 53.5, 261.3, 239.3, 102.7, 131.1, 69.0, 31.5, 139.3, 237.4, 216.8, 199.1, 109.8, 26.8, 129.4, 213.4, 16.9, 27.5, 120.5, 5.4, 116.0, 76.4, 239.8, 75.3, 68.4, 213.5, 193.2, 76.3, 110.7, 88.3, 109.8, 134.3, 28.6, 217.7, 250.9, 107.4, 163.3, 197.6, 184.9, 289.7, 135.2, 222.4, 296.4, 280.2, 187.9, 238.2, 137.9, 25.0, 90.4, 13.1, 255.4, 225.8, 241.7, 175.7, 209.6, 78.2, 75.1, 139.2, 76.4, 125.7, 19.4, 141.3, 18.8, 224.0, 123.1, 229.5, 87.2, 7.8, 80.2, 220.3, 59.6, 0.7, 265.2, 8.4, 219.8, 36.9, 48.3, 25.6, 273.7, 43.0, 184.9, 73.4, 193.7, 220.5, 104.6, 96.2, 140.3, 240.1, 243.2, 38.0, 44.7, 280.7, 121.0

In [27]:
r, _ = pearsonr(x, y)
print(r)

0.054808664465830145


---
## Univariate, Bivariate & Multivariate Plots

We won't draw plots (no matplotlib yet).

---
## 🎲  Introduction to Probability

**Probability** = How likely is an event to happen?

```
P(A) = Number of favorable outcomes / Total possible outcomes
```

Key rules:
- `0 ≤ P(A) ≤ 1`
- `P(A) + P(Aᶜ) = 1`   ← complement rule

In [29]:
sample_space = [1, 2, 3, 4, 5, 6]
event_six = []
for x in sample_space:
    if x ==6:
        event_six.append(x)
# event_six = [x for x in sample_space if x == 6]
event_six

[6]

In [30]:
# BASIC PROBABILITY — Dice example


# Sample space for a 6-sided die
sample_space = [1, 2, 3, 4, 5, 6]
total = len(sample_space)

# Events
event_six    = [x for x in sample_space if x == 6]
event_even   = [x for x in sample_space if x % 2 == 0]
event_gt4    = [x for x in sample_space if x > 4]
event_prime  = [x for x in sample_space if x in [2, 3, 5]]

print('Probability: Rolling a Fair Die')
print(f'Sample Space S = {sample_space}  (total outcomes = {total})')
print()
print(f'P(rolling a 6)       = {len(event_six)}/{total} = {len(event_six)/total:.4f}')
print(f'P(even number)       = {len(event_even)}/{total} = {len(event_even)/total:.4f}  events={event_even}')
print(f'P(greater than 4)    = {len(event_gt4)}/{total} = {len(event_gt4)/total:.4f}  events={event_gt4}')
print(f'P(prime number)      = {len(event_prime)}/{total} = {len(event_prime)/total:.4f}  events={event_prime}')
print()

# Complement
p_six = len(event_six) / total
print(f'Complement Rule: P(NOT 6) = 1 - P(6) = 1 - {p_six:.4f} = {1 - p_six:.4f}')

# Union
A = set(event_even)
B = set(event_gt4)
A_union_B = A | B
A_inter_B = A & B
p_union_formula = len(A)/total + len(B)/total - len(A_inter_B)/total

print()
print('--- Union & Intersection ---')
print(f'A = even numbers    = {sorted(A)}')
print(f'B = greater than 4  = {sorted(B)}')
print(f'A ∩ B (AND, both)   = {sorted(A_inter_B)}')
print(f'A ∪ B (OR, either)  = {sorted(A_union_B)}')
print()
print(f'P(A ∪ B) directly   = {len(A_union_B)}/{total} = {len(A_union_B)/total:.4f}')

Probability: Rolling a Fair Die
Sample Space S = [1, 2, 3, 4, 5, 6]  (total outcomes = 6)

P(rolling a 6)       = 1/6 = 0.1667
P(even number)       = 3/6 = 0.5000  events=[2, 4, 6]
P(greater than 4)    = 2/6 = 0.3333  events=[5, 6]
P(prime number)      = 3/6 = 0.5000  events=[2, 3, 5]

Complement Rule: P(NOT 6) = 1 - P(6) = 1 - 0.1667 = 0.8333

--- Union & Intersection ---
A = even numbers    = [2, 4, 6]
B = greater than 4  = [5, 6]
A ∩ B (AND, both)   = [6]
A ∪ B (OR, either)  = [2, 4, 5, 6]

P(A ∪ B) directly   = 4/6 = 0.6667


In [36]:
# LAW OF LARGE NUMBERS — Simulation with random
# The more coin flips, the closer P(heads) gets to 0.5

random.seed(42)

print('=== Law of Large Numbers: Coin Flip Simulation ===')
print()
print(f'{"Flips":>10}  {"Heads":>8}  {"P(Heads)":>10}  {"Error from 0.5":>16}')
print('-' * 50)

for n_flips in [10, 50, 100, 500, 1000, 5000, 10000, 100000]:
    flips = [random.randint(0, 1) for _ in range(n_flips)]  # 0=tails, 1=heads
    heads = sum(flips)
    p_heads = heads / n_flips
    error = abs(p_heads - 0.5)
    print(f'{n_flips:>10}  {heads:>8}  {p_heads:>10.4f}  {error:>16.4f}')

print()
print('As n increases, the estimated probability converges to the true value (0.5).')
print('   This is the foundation of why ML needs large datasets!')

=== Law of Large Numbers: Coin Flip Simulation ===

     Flips     Heads    P(Heads)    Error from 0.5
--------------------------------------------------
        10         2      0.2000            0.3000
        50        23      0.4600            0.0400
       100        44      0.4400            0.0600
       500       257      0.5140            0.0140
      1000       528      0.5280            0.0280
      5000      2525      0.5050            0.0050
     10000      4946      0.4946            0.0054
    100000     49937      0.4994            0.0006

As n increases, the estimated probability converges to the true value (0.5).
   This is the foundation of why ML needs large datasets!


### Excerise

> Note: For all the given practice questions, use `insurance.csv` Dataset.
1. Extract the features and output via file handling.
2. Find the Measure of Central Tendency for all the quantitative columns.
3. Find the Measure of Dispersion and Position for all the quantitative columns.
4. Find the corelation between each features and output. You can skip qualitative feature.
5. Find the corelation among all the features.
6. Find the corealtion between each feature and output on gender base (male, female).
7. Find the probability of male in dataset. Same do for female too.
8. Find the probability of persons whose age greater than 25 and then find their corelation with output.
9. Find the probability of person who have no children or are male.